# Stage 1 - Problem Identification & Data Overview: Evidence Notebook

Module: Databases and Analytics (CP6UA56O)
Name: Fathimath Zahra
Purpose: Verify the numeric claims used in Sections 1, 2, and 3 of the
final report, directly from the NorthStar dataset hosted on GitHub.

This notebook performs no analysis, it only **counts and inspects** the raw
data so that every figure cited in the report is traceable to a single
reproducible source.

In [3]:
# Step 1 : Load NorthStar Dataset directly from Github
import pandas as pd

#Base raw-content URL of the repository
BASE= ("https://raw.githubusercontent.com/fathimathzahrahei/northstar-databases-analytics_dba_coursework/main/data/raw/data/raw/")

#Load all the operational CSV files
files= ['hubs', 'customers', 'drivers', 'vehicles', 'orders', 'deliveries', 'incidents', 'complaints', 'app_events']
dfs= {name: pd.read_csv(BASE + name + '.csv') for name in files}

#Quick confirmation that every file loaded
for name, df in dfs.items():
  print(f"{name:<12} -> {len(df):>} rows x {df.shape[1]:>2} cols")

hubs         -> 8 rows x  5 cols
customers    -> 650 rows x  9 cols
drivers      -> 170 rows x  8 cols
vehicles     -> 120 rows x  8 cols
orders       -> 1250 rows x 11 cols
deliveries   -> 950 rows x 13 cols
incidents    -> 280 rows x  7 cols
complaints   -> 320 rows x 10 cols
app_events   -> 640 rows x 10 cols


## Evidence 1 - Fragmented categorical coding

The case study states:"internal systems remain fragmented, inconsis and increasingly difficult to manage." The cell below counts the distinct
spellings of each zone field across the six tables that record a zone. If
the data were consistent, each field would contain at most 7 unique values
(one per zone). Any value above 7 is direct evidence of inconsistent
categorical coding.


In [4]:
# Evidence 1 - Inconsistent zone coding across tables
zone_columns = {
    'orders.pickup_zone'         : dfs['orders']['pickup_zone'],
    'orders.dropoff_zone'        : dfs['orders']['dropoff_zone'],
    'customers.home_zone'        : dfs['customers']['home_zone'],
    'drivers.base_zone'          : dfs['drivers']['base_zone'],
    'vehicles.assigned_zone'     : dfs['vehicles']['assigned_zone'],
    'app_events.zone_context'    : dfs['app_events']['zone_context'],
}
print(f"{'Field':<32} {'Unique values':>14}")
print('-'*48)
for col, series in zone_columns.items():
  print(f"{col:<32} {series.nunique():>14}")
print("\nDistinct values found in orders.pickup_zone:")
print(sorted(dfs['orders']['pickup_zone'].dropna().unique()))

Field                             Unique values
------------------------------------------------
orders.pickup_zone                           16
orders.dropoff_zone                          16
customers.home_zone                          16
drivers.base_zone                            16
vehicles.assigned_zone                       16
app_events.zone_context                      16

Distinct values found in orders.pickup_zone:
['AIRPORT', 'Airport', 'CENTRAL', 'Central', 'Ctr', 'EAST', 'East', 'NORTH', 'North', 'RiverSide', 'Riverside', 'SOUTH', 'South', 'WEST', 'West', 'north']


# Evidence 2 - Missing values across operational tables

The case study claims that "more data than ever is being collected" yet
reporting systems cannot reveal where losses are really occurring.One
contributing factor is that operational tables contain non-trivial volumes
of missing values. The cell below quantifies the missing-data footprint of
each table.

In [7]:
#Evidence 2 - Missing value footprint
print(f"{'Table':<12} {'Column':<32} {'Missing':>8} {'Rate':>7}")
print('-'*64)
for name, df in dfs.items():
  miss= df.isna().sum()
  miss= miss[miss>0]
  for col, n in miss.items():
    rate= f"{n/len(df)*100:1f}%"
    print(f"{name:<12} {col:<32} {n:>8} {rate:>7}")

Table        Column                            Missing    Rate
----------------------------------------------------------------
customers    loyalty_score                          20 3.076923%
customers    preferred_channel                      13 2.000000%
drivers      training_score                          7 4.117647%
vehicles     battery_health_pct                      4 3.333333%
orders       booking_channel                        25 2.000000%
deliveries   delivery_completed_at                  19 2.000000%
deliveries   customer_rating_post_delivery          14 1.473684%
incidents    resolved_hours                         17 6.071429%
complaints   compensation_amount                    16 5.000000%
app_events   order_id                              144 22.500000%


## Evidence 3 - Logical impossibilities in the operational record

The case study notes that "transactions appear 'completed' in one system
and 'exception-handled' in another." The cell below checks whether the
delivery record is internally consistent: for each delivery, the
`delivery_completed_at` timestamp must logically come **after** the
`dispatch_time` timestamp. Any record where completion precedes dispatch
is impossible and indicates data-integrity failure between the dispatch
and completion systems.

In [10]:
#Evidence 3 - Deliveries completed before they were dispatched
d= dfs['deliveries'].copy()
d['dispatch_time']= pd.to_datetime(d['dispatch_time'], errors='coerce')
d['delivery_completed_at']= pd.to_datetime(d['delivery_completed_at'], errors='coerce')

impossible= (d['delivery_completed_at'] < d['dispatch_time']).sum()
total= d['delivery_completed_at'].notna().sum()

print(f"Total deliveries with completion timestamp : {total}")
print(f"Deliveries completed Before Dispatch : {impossible}")
print(f"Logical-failure rate : {impossible/total*100:.1f}%")

Total deliveries with completion timestamp : 931
Deliveries completed Before Dispatch : 64
Logical-failure rate : 6.9%


## Evidence 4 - Hidden complaint signal masked by 'OnTime' status

The customer-experience director argues that "service reliability is
falling because the organisation is not properly connecting complaints,
missed journeys, failed deliveries and driver incidents into one view."
The cell below tests this directly by joining deliveries to complaints
on `order_id` and counting how many deliveries marked `OnTime` actually
have a customer complaint attached.

In [11]:
# Evidence 4 - Deliveries marked OnTime that recieved a complaint
joined = dfs['deliveries'].merge(
    dfs['complaints'][['order_id', 'complaint_type', 'severity']],
    on='order_id',
    how='inner'
)
status_counts = joined['delivery_status'].value_counts()
print("Delivery status distribution among complained-about orders:")
print(status_counts)

ontime_with_complaint = (joined['delivery_status']== 'OnTime').sum()
total_complained = len(joined)
print(f"\n'OnTime' deliveries that trigged a complaint:"
f"{ontime_with_complaint} / {total_complained}"
f"({ontime_with_complaint/total_complained*100:.1f}%)")

Delivery status distribution among complained-about orders:
delivery_status
OnTime     149
Delayed     48
Failed      35
Name: count, dtype: int64

'OnTime' deliveries that trigged a complaint:149 / 232(64.2%)


## Evidence 5 - Manual route override behaviour

The operations director attributes problems to "poor route allocation",
while there is also concern that drivers may use overrides to "avoid
performance targets". The cell below characterises override behaviour
across the delivery population.

In [12]:
# Evidence 5 - Manual route override profile
mr = dfs['deliveries']['manual_route_override_count']
print(f"Mean overrides per delivery : {mr.mean():.2f}")
print(f"Median overrides per delivery : {mr.median():.0f}")
print(f"Maximum overrides on any delivery : {mr.max()}")
print(f"Deliveries with >= 3 overrides : {(mr>=3).sum()}")
f"({(mr>=3).mean()*100:.1f}% of all deliveries)"
print("\nOverride count vs delivery status:")
print(dfs['deliveries'].groupby('delivery_status')['manual_route_override_count'].mean().round(2))

Mean overrides per delivery : 0.97
Median overrides per delivery : 1
Maximum overrides on any delivery : 7
Deliveries with >= 3 overrides : 88

Override count vs delivery status:
delivery_status
Delayed    1.07
Failed     1.04
OnTime     0.92
Name: manual_route_override_count, dtype: float64


## Evidence 6 - Overall delivery performance

The board has noticed "delays are becoming more frequent". The cell
below establishes the baseline: across the full delivery population,
what proportion succeed, fail, or are delayed?

In [13]:
# Evidence 6 - Aggregate delivery outcome rates
status = dfs['deliveries']['delivery_status'].value_counts(normalize=True * 100)
counts = dfs['deliveries']['delivery_status'].value_counts()

print("Delivery outcome distribution:")
for s in status.index:
  print(f"{s:<10} {counts[s]:>5} ({status[s]:.1f}%)")
failure_or_delay = ((dfs['deliveries']['delivery_status'].isin(['Failed','Delayed'])).mean())*100
print(f"\nFailure-or-delay rate : {failure_or_delay:.1f}%")


Delivery outcome distribution:
OnTime       616 (0.6%)
Delayed      202 (0.2%)
Failed       132 (0.1%)

Failure-or-delay rate : 35.2%
